## Colab training + reporting (clean)\n\n### Enable GPU (Colab UI)\nColab can’t be forced to GPU from code. Do:\n- `Runtime` → `Change runtime type` → **Hardware accelerator = GPU** → `Save`\n\n### What this notebook does\n- (Optional) trains all 5 models using `experiments/train.py`\n- Saves `results/{model}_results.npz` + `results/checkpoints/{model}.pt`\n- Plots **ECE over time** and prints a compact metrics table\n

In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nimport torch\n\n# In Colab, clone once (edit URL if needed):\n# %cd /content\n# !git clone <YOUR_REPO_URL> Calibration-Confidence\n\nREPO_DIR = os.environ.get("REPO_DIR", "/content/Calibration-Confidence")\nif Path(REPO_DIR).is_dir():\n    os.chdir(REPO_DIR)\nelse:\n    # Local fallback: assume notebook is in repo_root/notebooks\n    os.chdir(Path.cwd().parent)\n\nif not Path("experiments/train.py").is_file():\n    raise FileNotFoundError(\n        "Repo root not found. In Colab, clone/copy the repo to /content/Calibration-Confidence "\n        "(or set REPO_DIR env var), then re-run this cell."\n    )\n\nROOT = os.path.abspath(os.getcwd())\nif ROOT not in sys.path:\n    sys.path.insert(0, ROOT)\n\nos.makedirs("results/checkpoints", exist_ok=True)\nos.makedirs("results/figures", exist_ok=True)\nos.makedirs("results", exist_ok=True)\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\nprint("Repo root:", os.getcwd())\nprint("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| Device:", device)\nif torch.cuda.is_available():\n    print("GPU:", torch.cuda.get_device_name(0))\n

In [ ]:
# Optional: train all 5 models\n# Flip RUN_TRAINING=True only if results/*.npz don't exist yet.\n\nimport subprocess\n\nRUN_TRAINING = False\n\nEPOCHS = 10\nSEQ_LEN = 20\nBATCH_SIZE = 32\nLR = 1e-3\n\nmodels = ["mlp", "deep", "rnn", "lstm", "residual"]\n\ndef run(args: list[str]):\n    r = subprocess.run(args, text=True, capture_output=True)\n    if r.returncode != 0:\n        print(r.stdout)\n        print(r.stderr)\n        raise RuntimeError(f"Command failed (exit {r.returncode}): {' '.join(args)}")\n\nif RUN_TRAINING:\n    for m in models:\n        run([\n            sys.executable, "experiments/train.py",\n            "--model", m,\n            "--epochs", str(EPOCHS),\n            "--seq-len", str(SEQ_LEN),\n            "--batch-size", str(BATCH_SIZE),\n            "--lr", str(LR),\n            "--no-resume",\n            "--checkpoint-path", f"results/checkpoints/{m}.pt",\n            "--results-path", f"results/{m}_results.npz",\n        ])\n    print("Done training all models.")\nelse:\n    print("Skipping training (RUN_TRAINING=False).")\n

In [ ]:
# Plot ECE over time for all models\n\nfrom experiments.plot_ece_over_time import plot_ece_over_time\n\nnpz_paths = [\n    "results/mlp_results.npz",\n    "results/deep_results.npz",\n    "results/rnn_results.npz",\n    "results/lstm_results.npz",\n    "results/residual_results.npz",\n]\nlabels = ["mlp", "deep", "rnn", "lstm", "residual"]\n\nplot_ece_over_time(\n    npz_paths=npz_paths,\n    labels=labels,\n    save_path="results/figures/ece_over_time_all.png",\n)\n

In [ ]:
# Compact summary table from each results .npz\n\nimport os\nimport numpy as np\n\ntry:\n    import pandas as pd\n    PANDAS_AVAILABLE = True\nexcept Exception:\n    pd = None\n    PANDAS_AVAILABLE = False\n\nruns = [\n    ("mlp", "results/mlp_results.npz"),\n    ("deep", "results/deep_results.npz"),\n    ("rnn", "results/rnn_results.npz"),\n    ("lstm", "results/lstm_results.npz"),\n    ("residual", "results/residual_results.npz"),\n]\n\nrows = []\nfor name, path in runs:\n    if not os.path.isfile(path):\n        rows.append({"model": name, "path": path, "status": "missing"})\n        continue\n    d = np.load(path)\n    row = {"model": name, "path": path, "status": "ok"}\n    if "val_loss" in d:\n        row["final_val_loss"] = float(np.asarray(d["val_loss"])[-1])\n    if "ece_over_time" in d:\n        row["final_ece"] = float(np.asarray(d["ece_over_time"])[-1])\n    if "per_example_loss" in d:\n        row["mean_per_example_loss"] = float(np.mean(np.asarray(d["per_example_loss"]).ravel()))\n    rows.append(row)\n\nif PANDAS_AVAILABLE:\n    df = pd.DataFrame(rows)\n    cols = [c for c in ["model", "status", "final_val_loss", "final_ece", "mean_per_example_loss", "path"] if c in df.columns]\n    display(df[cols])\nelse:\n    for r in rows:\n        print(r)\n